In [31]:
from pymilvus import connections, utility, MilvusException
connections.connect(host="localhost", port="19530")
try:
    collections = utility.list_collections()
    print("List of collections: ", collections)
except MilvusException as e:
    print(e)



from pymilvus import MilvusClient, DataType
CLUSTER_ENDPOINT = "http://localhost:19530"
TOKEN = "root:Milvus"
client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)

List of collections:  ['aic24_clip']


In [3]:
import os
import numpy as np
import pandas as pd


CLIP_EMBEDDINGS_DIR = "/root/LSC24_SemanticSearchWebApp/backend/data/embeddings/clip"
METADATA_PATH = "/root/LSC24_SemanticSearchWebApp/backend/data/metadata/metadata_v6.csv"
metadata_df = pd.read_csv(METADATA_PATH)

/tmp/ipykernel_45464/2055195050.py:8: DtypeWarning: Columns (13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(METADATA_PATH)


In [4]:
metadata_df.rename(columns={"Unnamed: 0": "image_link"}, inplace=True)
metadata_df.set_index("image_link", inplace=True)

In [12]:
len(os.listdir(CLIP_EMBEDDINGS_DIR))

726

In [4]:
MISSING_FILES = ["L01_V001.npy", "L07_V015.npy", "L08_V001.npy", "L08_V002.npy", "L09_V005.npy", "L09_V006.npy"]

In [10]:
# load image names from /home/pc/LSC24_SemanticSearchWebApp/backend/data/aic24/metadata/missing_video_ids.txt 
# add them to image_link column in metadata_df

missing_video_ids_path = "/home/pc/LSC24_SemanticSearchWebApp/backend/data/aic24/metadata/missing_video_ids.txt"
with open(missing_video_ids_path, "r") as f:
    missing_image_paths = f.readlines()
    missing_image_paths = [path.strip() for path in missing_image_paths]
    print(missing_image_paths)
    for path in missing_image_paths:
        metadata_df.loc[len(metadata_df)] = {"image_link": path}

['L07/L07_V015/0001.webp', 'L07/L07_V015/0002.webp', 'L07/L07_V015/0005.webp', 'L07/L07_V015/0009.webp', 'L07/L07_V015/0013.webp', 'L07/L07_V015/0017.webp', 'L07/L07_V015/0018.webp', 'L07/L07_V015/0019.webp', 'L07/L07_V015/0021.webp', 'L07/L07_V015/0022.webp', 'L07/L07_V015/0023.webp', 'L07/L07_V015/0024.webp', 'L07/L07_V015/0025.webp', 'L07/L07_V015/0026.webp', 'L07/L07_V015/0027.webp', 'L07/L07_V015/0028.webp', 'L07/L07_V015/0030.webp', 'L07/L07_V015/0032.webp', 'L07/L07_V015/0033.webp', 'L07/L07_V015/0034.webp', 'L07/L07_V015/0035.webp', 'L07/L07_V015/0037.webp', 'L07/L07_V015/0039.webp', 'L07/L07_V015/0040.webp', 'L07/L07_V015/0041.webp', 'L07/L07_V015/0042.webp', 'L07/L07_V015/0045.webp', 'L07/L07_V015/0049.webp', 'L07/L07_V015/0053.webp', 'L07/L07_V015/0057.webp', 'L07/L07_V015/0061.webp', 'L07/L07_V015/0065.webp', 'L07/L07_V015/0066.webp', 'L07/L07_V015/0068.webp', 'L07/L07_V015/0071.webp', 'L07/L07_V015/0074.webp', 'L07/L07_V015/0078.webp', 'L07/L07_V015/0082.webp', 'L07/L07_V0

In [19]:
metadata_df['video_id'] = metadata_df['image_link'].apply(lambda x: x.split("/")[1])
metadata_df['frame_id'] = metadata_df['image_link'].apply(lambda x: (int(x.split("/")[2].split(".")[0]) - 1) * 25.0)

In [20]:
metadata_df.iloc[len(metadata_df)-2200:len(metadata_df)]

,Unnamed: 0,image_link,video_id,frame_id,scene_id,scene_begin_frame_id,scene_end_frame_id,video_url,timestamp,ocr,...,context_start,context_end,context_vi,context_en,context_en_keywords,context_id,object_global_encoding,object_local_encoding,color_global_encoding,color_local_encoding
175405,NaN,L07/L07_V015/0716.webp,L07_V015,17875.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175406,NaN,L07/L07_V015/0719.webp,L07_V015,17950.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175407,NaN,L07/L07_V015/0723.webp,L07_V015,18050.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175408,NaN,L07/L07_V015/0726.webp,L07_V015,18125.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175409,NaN,L07/L07_V015/0729.webp,L07_V015,18200.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177600,NaN,L09/L09_V006/1469.webp,L09_V006,36700.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177601,NaN,L09/L09_V006/1473.webp,L09_V006,36800.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177602,NaN,L09/L09_V006/1477.webp,L09_V006,36900.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177603,NaN,L09/L09_V006/1479.webp,L09_V006,36950.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
PREFIXES = ["L13", "L14", "L15", "L16", "L17", "L18", "L19", "L20", "L21", "L22", "L23", "L24"]

for ebd_file in sorted(os.listdir(CLIP_EMBEDDINGS_DIR)):
    prefix = ebd_file.split("_")[0]
    # if prefix not in PREFIXES:
    #     continue
# for ebd_file in MISSING_FILES:
    if ebd_file.endswith('.npy'):
        ebd_path = f"{CLIP_EMBEDDINGS_DIR}/{ebd_file}"
        video_id = ebd_file[:-4]
        embeddings = np.load(ebd_path)
        image_links = metadata_df[metadata_df['video_id'] == video_id].index.to_list()
        if embeddings.shape[0] == len(image_links):            # if ebd is found but no metadata found then the data is invalid -> no upload
            data = []
            print(image_links)
            for i in range(len(image_links)):
                data.append({
                    "url": image_links[i],
                    "embedding": embeddings[i].astype('float32')
                })
            res = client.insert(collection_name="aic24_clip", data=data)
            print(res)            
            print(video_id)
            print(len(data))
            print()
        else:
            print(f"Invalid data: {video_id}")
            print(f"{embeddings.shape[0]}\t{len(image_links)}")
            print()

RPC error: [insert_rows], <DataNotMatchException: (code=1, message=The data in the same column must be of the same type.)>, <Time:{'RPC start': '2024-09-23 10:41:25.769118', 'RPC error': '2024-09-23 10:41:25.771612'}>
Failed to insert batch starting at entity: 0/502


['L01/L01_V001/0002.webp', 'L01/L01_V001/0005.webp', 'L01/L01_V001/0009.webp', 'L01/L01_V001/0013.webp', 'L01/L01_V001/0017.webp', 'L01/L01_V001/0018.webp', 'L01/L01_V001/0019.webp', 'L01/L01_V001/0021.webp', 'L01/L01_V001/0023.webp', 'L01/L01_V001/0026.webp', 'L01/L01_V001/0027.webp', 'L01/L01_V001/0029.webp', 'L01/L01_V001/0031.webp', 'L01/L01_V001/0033.webp', 'L01/L01_V001/0034.webp', 'L01/L01_V001/0036.webp', 'L01/L01_V001/0039.webp', 'L01/L01_V001/0043.webp', 'L01/L01_V001/0047.webp', 'L01/L01_V001/0051.webp', 'L01/L01_V001/0055.webp', 'L01/L01_V001/0057.webp', 'L01/L01_V001/0061.webp', 'L01/L01_V001/0065.webp', 'L01/L01_V001/0067.webp', 'L01/L01_V001/0069.webp', 'L01/L01_V001/0071.webp', 'L01/L01_V001/0074.webp', 'L01/L01_V001/0078.webp', 'L01/L01_V001/0081.webp', 'L01/L01_V001/0085.webp', 'L01/L01_V001/0089.webp', 'L01/L01_V001/0093.webp', 'L01/L01_V001/0096.webp', 'L01/L01_V001/0097.webp', 'L01/L01_V001/0098.webp', 'L01/L01_V001/0099.webp', 'L01/L01_V001/0100.webp', 'L01/L01_V0

DataNotMatchException: <DataNotMatchException: (code=1, message=The data in the same column must be of the same type.)>